# Week 36, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w036d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
T = {("off", "on"): "idle", ("idle", "go"): "busy", ("busy", "done"): "idle"}
state = "off"
log = []
for event in ["go", "on", "go", "done", "off"]:
    if (state, event) in T:
        state = T[(state, event)]
        log.append(state)
print(state, log)

**Predict before you run:** what will this print?



In [ ]:
undo, redo = [], []
value = 0
for n in [5, 3, 2]:
    value += n
    undo.append(n)
    redo.clear()
n = undo.pop(); value -= n; redo.append(n)
n = undo.pop(); value -= n; redo.append(n)
n = redo.pop(); value += n; undo.append(n)
print(value, undo, redo)

**Predict before you run:** what will this print?



In [ ]:
class Count:
    def __init__(self, n):
        self.n = n
    def __iter__(self):
        return iter(range(self.n))

c = Count(3)
it = iter(c)
print(next(it), list(it), list(c), list(it))

**Predict before you run:** what will this print?



In [ ]:
def build(env):
    repo = {"memory": "Memory", "json": "Json"}[env.get("STORE", "memory")]
    return f"Service({repo})"

print(build({}), build({"STORE": "json"}))

**Predict before you run:** what will this print?



In [ ]:
class Repo:
    def __init__(self):
        self.d = {}
    def save(self, k, v):
        self.d[k] = v

class Service:
    def __init__(self, repo):
        self.repo = repo
    def put(self, k, v):
        if k in self.repo.d:
            raise KeyError(k)
        self.repo.save(k, v)

s = Service(Repo())
s.put("a", 1)
try:
    s.put("a", 2)
except KeyError as err:
    print("KeyError", err, s.repo.d)

### Exercise: A door with a lock

Write a transition table `DOOR` and a class `Door` (starting `"closed"`) with `handle(event)`. States and events: closed + open → opened; opened + close → closed; closed + lock → locked; locked + unlock → closed. Anything else raises `ValueError` listing the allowed events (sorted). `handle` returns the new state.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_door():\n    \"a door\"\n    D = ns[\"Door\"]\n    d = D()\n    assert [d.handle(e) for e in [\"lock\", \"unlock\", \"open\", \"close\"]] == [\"locked\", \"closed\", \"opened\", \"closed\"]\n    d.handle(\"lock\")\n    try:\n        d.handle(\"open\")\n        assert False, \"a locked door can't open\"\n    except ValueError as err:\n        assert \"unlock\" in str(err)\n    assert d.state == \"locked\" and len(DOOR) == 4")

### Exercise: An iterable of chunks

Write an iterable `Chunks(text, size)` that gives the text in pieces of `size` characters (the last may be shorter), with a fresh iterator each time it's looped over (write the iterator class `ChunkIterator` too). A size below 1 raises `ValueError`.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_chunks():\n    \"chunks\"\n    C, CI = ns[\"Chunks\"], ns[\"ChunkIterator\"]\n    c = C(\"abcdefg\", 3)\n    assert list(c) == [\"abc\", \"def\", \"g\"] and list(c) == [\"abc\", \"def\", \"g\"] and isinstance(iter(c), CI) and list(C(\"\", 2)) == []\n    try:\n        C(\"x\", 0)\n        assert False\n    except ValueError:\n        pass")

### Exercise: Forge: a task board

Write the task board to the specification above.

In [ ]:
from dataclasses import dataclass, replace
from enum import Enum

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_domain_and_repo():\n    \"domain and repository\"\n    S, T, M = ns[\"Status\"], ns[\"Task\"], ns[\"MemoryTasks\"]\n    assert [s.value for s in S] == [\"todo\", \"doing\", \"done\", \"archived\"] and FLOW[(S.TODO, \"start\")] is S.DOING and len(FLOW) == 5\n    r = M()\n    r.add(T(1, \"A\"))\n    t = r.get(1)\n    t.title = \"changed\"\n    assert r.get(1).title == \"A\" and r.get(9) is None\n    r.save(t)\n    assert [x.title for x in r.all()] == [\"changed\"]\n    r.delete(1)\n    assert r.all() == []\n    assert all(issubclass(ns[e], ns[\"TaskError\"]) for e in [\"NoSuchTask\", \"BadMove\"])\ndef test_board():\n    \"the service\"\n    S = ns[\"Status\"]\n    b = build()\n    assert b.create(\"Write report\") == 1 and b.create(\"Book room\") == 2 and b.move(1, \"start\") is S.DOING\n    b.assign(1, \"Ada\")\n    assert b.columns() == {\"todo\": [\"Book room\"], \"doing\": [\"Write report\"]}\n    assert b.undo() and b.repo.get(1).assignee is None and b.undo() and b.repo.get(1).status is S.TODO\n    assert b.redo() and b.repo.get(1).status is S.DOING and b.redo() and b.repo.get(1).assignee == \"Ada\" and not b.redo()\n    b.undo()\n    b.undo()\n    b.undo()\n    assert b.columns() == {\"todo\": [\"Write report\"]} and b.create(\"New\") == 2\n    for f, err in [(lambda: b.move(1, \"finish\"), \"BadMove\"), (lambda: b.move(7, \"start\"), \"NoSuchTask\"), (lambda: b.create(\"  \"), \"TaskError\")]:\n        try:\n            f()\n            assert False, f\"should raise {err}\"\n        except ns[err]:\n            pass\n    assert b.columns() == {\"todo\": [\"Write report\", \"New\"]}, \"failed commands change nothing\"\ndef test_handle():\n    \"the presentation\"\n    b = build()\n    lines = [\"add Write report\", \"add Book room\", \"start 1\", \"assign 1 Ada\", \"finish 1\", \"show\", \"undo\", \"show\", \"redo\",\n             \"archive 1\", \"show\", \"assign 2 -\", \"start 9\", \"finish 2\", \"dance\"] + [\"undo\"] * 8 + [\"show\", \"redo\"]\n    got = [handle(l, b) for l in lines]\n    want = [\"added 1\", \"added 2\", \"1 is now doing\", \"1 assigned to Ada\", \"1 is now done\", \"todo: Book room\\ndone: Write report\",\n            \"undone\", \"todo: Book room\\ndoing: Write report\", \"redone\", \"1 is now archived\", \"todo: Book room\\narchived: Write report\",\n            \"2 unassigned\", \"error: no task 9\", None, \"unknown command\"] + [\"undone\"] * 7 + [\"nothing to undo\", \"empty board\", \"redone\"]\n    for line, g, w in zip(lines, got, want):\n        if w is None:\n            assert g.startswith(\"error: \"), f\"{line!r}: got {g!r}\"\n        else:\n            assert g == w, f\"{line!r}: got {g!r}, expected {w!r}\"")

> **Each piece in its place.** Follow one typed line through the program: `handle` turns `finish 1` into `board.move(1, "finish")`; the service wraps it in a `Move` command and runs it; the command loads the task from the repository, asks the domain's `FLOW` whether the move is allowed, saves it, and remembers how to undo it. No piece knows more than it needs: the presentation knows nothing about `FLOW`, the domain knows nothing about undo, and none of them knows the tasks live in a dictionary. Swapping the repository for a file, the presentation for a web page, or adding a new event to `FLOW` would each touch one place. That's what this whole phase has been building towards.

### Exercise: Forge bonus: a music player

Build a small music player:

- **State**: a table `PLAYER` from `(state, event)` to a new state, for the states `"stopped"`, `"playing"` and `"paused"`: stopped + play → playing; playing + pause → paused; paused + play → playing; playing + stop → stopped; paused + stop → stopped. `Player.press(event)` changes state (or raises `ValueError`) and returns the new state.
- **Queue with undo**: `Player` has a `queue` (a list of titles) changed only through commands: `add(title)`, `remove(index)` (an `IndexError` for a bad index) and `move(old_index, new_index)`, each undoable with `undo()` (which returns `False` if there's nothing to undo).
- **Iterator**: `Player.upcoming(repeat=False)` returns an iterator over the queue's titles. With `repeat=True` it starts again from the beginning after the last title, forever (an empty queue gives nothing, even with repeat). It's a snapshot: changing the queue afterwards doesn't affect an iterator already made.

In [ ]:
from itertools import islice

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_player():\n    \"states\"\n    P = ns[\"Player\"]\n    p = P()\n    assert [p.press(e) for e in [\"play\", \"pause\", \"play\", \"stop\"]] == [\"playing\", \"paused\", \"playing\", \"stopped\"]\n    for e in [\"pause\", \"stop\"]:\n        try:\n            p.press(e)\n            assert False, f\"{e} when stopped should raise ValueError\"\n        except ValueError:\n            pass\n    assert p.state == \"stopped\" and len(PLAYER) == 5\ndef test_queue():\n    \"the queue and undo\"\n    P = ns[\"Player\"]\n    p = P()\n    for t in [\"Ye\", \"Essence\", \"Calm Down\", \"Peru\"]:\n        p.add(t)\n    p.move(3, 0)\n    p.remove(2)\n    assert p.queue == [\"Peru\", \"Ye\", \"Calm Down\"]\n    assert p.undo() and p.queue == [\"Peru\", \"Ye\", \"Essence\", \"Calm Down\"] and p.undo() and p.queue == [\"Ye\", \"Essence\", \"Calm Down\", \"Peru\"]\n    try:\n        p.remove(10)\n        assert False\n    except IndexError:\n        pass\n    while p.undo():\n        pass\n    assert p.queue == [] and not p.undo()\ndef test_upcoming():\n    \"the iterator\"\n    P = ns[\"Player\"]\n    p = P()\n    for t in [\"A\", \"B\", \"C\"]:\n        p.add(t)\n    it = p.upcoming()\n    p.add(\"D\")\n    assert list(it) == [\"A\", \"B\", \"C\"] and list(islice(p.upcoming(repeat=True), 7)) == [\"A\", \"B\", \"C\", \"D\", \"A\", \"B\", \"C\"]\n    assert list(P().upcoming(repeat=True)) == [] and iter(it) is it")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [State](w036d1.html), [Command and undo](w036d2.html), [Iterator](w036d3.html), [layered architecture](w036d4.html) or [dependency injection](w036d5.html).

## Mixed review

## Forge: a task board

A team keeps its work on a board: tasks move from to-do, to doing, to done, and every change can be undone. Here's the specification, layer by layer.

**Domain.**

- `Status`, an Enum: `TODO = "todo"`, `DOING = "doing"`, `DONE = "done"`, `ARCHIVED = "archived"`.
- `FLOW`, a transition table from `(Status, event)` to `Status`: todo + `start` → doing; doing + `pause` → todo; doing + `finish` → done; done + `reopen` → todo; done + `archive` → archived.
- `Task`, a dataclass: `task_id` (int), `title`, `status` (default `Status.TODO`) and `assignee` (default `None`).
- `TaskError`, with `NoSuchTask` and `BadMove` as kinds of it.

**Infrastructure.** `MemoryTasks`, a repository with `add(task)`, `get(task_id)` (a **copy**, or `None`), `save(task)`, `delete(task_id)` and `all()` (copies, by id).

**Commands** (each with `execute()` and `undo()`, working through the repository):

- `Create(repo, title)`: adds a task with the next id (1 more than the largest ever used by this repository's tasks, starting at 1) and remembers it; undo deletes it. A blank title raises `TaskError`.
- `Move(repo, task_id, event)`: applies `FLOW`, remembering the old status; an unknown task raises `NoSuchTask`, an event not allowed raises `BadMove`; undo restores the old status.
- `Assign(repo, task_id, person)`: sets the assignee (`None` to unassign), remembering the old one.

**Application.** `Board(repo)`, the service: `create(title)` (returns the new id), `move(task_id, event)` (returns the new `Status`), `assign(task_id, person)`, `undo()` and `redo()` (return `True` or `False`, as in Day 2's `History`), all done through commands so every change can be undone; and `columns()`, a dictionary from each `Status`'s value, in the Enum's order, to the titles of its tasks, by id, leaving out empty statuses.

**Presentation.** `handle(line, board)`, returning text:

| Line | Result |
|---|---|
| `add Write report` | `added 1` |
| `start 1` (or any event) | `1 is now doing` |
| `assign 1 Ada` / `assign 1 -` | `1 assigned to Ada` / `1 unassigned` |
| `undo` / `redo` | `undone` / `redone`, or `nothing to undo` / `nothing to redo` |
| `show` | one line per non-empty column, like `todo: Write report, Book room` (or `empty board`) |

A `TaskError` becomes `error: ` and its message; anything else unrecognised gives `unknown command`.

**Composition root.** `build()` returns a `Board` with a new `MemoryTasks`.

## Forge bonus: a music player

## Weekly quiz

## Week 36 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can write state machines with state classes or a transition table, and reject events that aren't allowed. ([Day 1](w036d1.html))
- I can write undoable commands, undo and redo with two stacks, and all-or-nothing macros. ([Day 2](w036d2.html))
- I can explain iterables and iterators, and write iterator classes, including for structures like trees. ([Day 3](w036d3.html))
- I can organise a program in layers, with repositories, services and a thin presentation, keeping the dependency rule. ([Day 4](w036d4.html))
- I can inject dependencies, wire a program in a composition root from settings, and avoid hidden dependencies. ([Day 5](w036d5.html))
- I built a task board and a music player using all of it. (Today)

Tomorrow is a **rest day**. Next week: **robust programs**: exception hierarchies, custom exceptions, EAFP and LBYL, logging, and defensive programming.